# Section 1: Data Verification

| # | Figure | Description |
|---|--------|-------------|
| 1 | Trial inventory | Original vs preprocessed trial counts, missing blocks, split distribution |
| 2 | Trial count summary | Bar chart of DBS-OFF / DBS-ON trial counts per session |
| 3-6 | PSD DBS comparison | Power spectral density per ECoG channel, 4 sessions |
| 7 | Tracing speed DBS comparison | Mean velocity & acceleration traces by DBS condition |

In [1]:
import sys, os
os.chdir('/home/bobby/repos/latent-neural-dynamics-modeling')
sys.path.insert(0, '.')

from pathlib import Path
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import welch

from dashboard.thesis.constants import (
    COLOR_DBS_OFF, COLOR_DBS_ON, COLOR_PSID, COLOR_CHANCE,
    FONT_FAMILY, FONT_SIZE_BASE, FONT_SIZE_LABEL, FONT_SIZE_TICK,
    ThesisTheme, apply_thesis_style, grid_color, paper_colors, true_line_color,
    d_score_axis_label, rmse_axis_label,
)
from dashboard.thesis.loaders import load_split_results, channels_as_str_list
from dashboard.thesis.plot_config import RESULTS_ROOT, get_triplets, load_results_for_triplet
from utils.thesis_result_timestamps import latest_run_timestamp_on_disk

OUT = Path('thesis_figures/sec1'); OUT.mkdir(parents=True, exist_ok=True)
results_root = Path('results').resolve()

## Trial inventory

For each session: original blocks/trials from `participants.parquet`,
preprocessed trials in each split, and which blocks/trials were removed.

In [2]:
SESSIONS = [
    ('PDI1_S2', 'PDI1', 2, 'psid_behavioral_PDI1_2_nx_25_n2_i50_dbs_both_200Hz_narrow_band'),
    ('PDI1_S4', 'PDI1', 4, 'psid_behavioral_PDI1_4_nx_15_n2_i50_dbs_both_200Hz_narrow_band'),
    ('PDI4_S2', 'PDI4', 2, 'psid_behavioral_PDI4_2_nx_30_n6_i50_dbs_both_200Hz_narrow_band'),
    ('PDI4_S3', 'PDI4', 3, 'psid_behavioral_PDI4_3_nx_25_n6_i50_dbs_both_200Hz_narrow_band'),
]

orig = pl.read_parquet('data/participants.parquet', hive_partitioning=True)

rows = []
for label, pid, sess, variant in SESSIONS:
    # Original data
    sub = orig.filter((pl.col('participant_id') == pid) & (pl.col('session') == sess)).sort('block')
    orig_blocks = sub['block'].to_list()
    orig_trials = {int(r['block']): len(r['trials']) for r in sub.iter_rows(named=True)}
    orig_dbs = {int(r['block']): 'ON' if r['dbs_stim'] == 1 else 'OFF' for r in sub.iter_rows(named=True)}
    orig_total = sum(orig_trials.values())

    # Split data
    base = results_root / variant / 'split'
    split_counts = {}
    split_block_trials = {}
    for split in ['train', 'val', 'test']:
        df = pl.read_parquet(base / f'{split}.parquet')
        split_counts[split] = len(df)
        for r in df.group_by('block').len().iter_rows(named=True):
            b = int(r['block'])
            split_block_trials.setdefault(b, {})[split] = r['len']

    split_total = sum(split_counts.values())
    split_blocks = sorted(split_block_trials.keys())

    # Missing blocks (in original but not in splits)
    missing_blocks = [b for b in orig_blocks if b not in split_blocks]

    # Removed trials per block
    removed_per_block = {}
    for b in orig_blocks:
        orig_n = orig_trials[b]
        split_n = sum(split_block_trials.get(b, {}).values())
        if orig_n != split_n:
            removed_per_block[b] = orig_n - split_n

    rows.append({
        'Session': label,
        'Original blocks': len(orig_blocks),
        'Original trials': orig_total,
        'Missing blocks': str(missing_blocks) if missing_blocks else '-',
        'Preprocessed trials': split_total,
        'Removed trials': orig_total - split_total,
        'Train': split_counts['train'],
        'Val': split_counts['val'],
        'Test': split_counts['test'],
    })

    # Print detailed block info
    print(f"\n{label}: {len(orig_blocks)} blocks, {orig_total} original trials -> {split_total} preprocessed")
    if missing_blocks:
        print(f"  Missing blocks: {missing_blocks}")
    if removed_per_block:
        for b, n in sorted(removed_per_block.items()):
            print(f"  Block {b} (DBS-{orig_dbs[b]}): {orig_trials[b]} -> {orig_trials[b]-n} ({n} removed)")

print(
    "\nTrial inventory summary — compares the raw "
    "data/participants.parquet record (per block/trial counts as collected at the bedside) "
    "against what made it into the train/val/test parquet splits under "
    "results/<variant>/split/. Removed trials = "
    "preprocessing rejections (motion, missing samples, etc.)."
)

# Summary table
summary = pl.DataFrame(rows)
summary


PDI1_S2: 12 blocks, 143 original trials -> 144 preprocessed
  Block 11 (DBS-OFF): 11 -> 12 (-1 removed)



PDI1_S4: 10 blocks, 111 original trials -> 106 preprocessed
  Block 6 (DBS-ON): 11 -> 10 (1 removed)
  Block 7 (DBS-OFF): 11 -> 8 (3 removed)
  Block 8 (DBS-ON): 11 -> 10 (1 removed)
  Block 9 (DBS-OFF): 8 -> 7 (1 removed)
  Block 10 (DBS-ON): 10 -> 11 (-1 removed)



PDI4_S2: 10 blocks, 116 original trials -> 119 preprocessed
  Block 5 (DBS-ON): 12 -> 11 (1 removed)
  Block 6 (DBS-OFF): 11 -> 12 (-1 removed)
  Block 11 (DBS-ON): 10 -> 12 (-2 removed)
  Block 13 (DBS-ON): 11 -> 12 (-1 removed)



PDI4_S3: 11 blocks, 110 original trials -> 117 preprocessed
  Missing blocks: [7]
  Block 3 (DBS-OFF): 9 -> 12 (-3 removed)
  Block 4 (DBS-ON): 11 -> 12 (-1 removed)
  Block 5 (DBS-OFF): 6 -> 12 (-6 removed)
  Block 7 (DBS-OFF): 7 -> 0 (7 removed)
  Block 11 (DBS-OFF): 7 -> 11 (-4 removed)

Trial inventory summary — compares the raw data/participants.parquet record (per block/trial counts as collected at the bedside) against what made it into the train/val/test parquet splits under results/<variant>/split/. Removed trials = preprocessing rejections (motion, missing samples, etc.).


Session,Original blocks,Original trials,Missing blocks,Preprocessed trials,Removed trials,Train,Val,Test
str,i64,i64,str,i64,i64,i64,i64,i64
"""PDI1_S2""",12,143,"""-""",144,-1,86,14,44
"""PDI1_S4""",10,111,"""-""",106,5,63,11,32
"""PDI4_S2""",10,116,"""-""",119,-3,71,12,36
"""PDI4_S3""",11,110,"""[7]""",117,-7,70,11,36


## Fig 1: Trial count summary (DBS-OFF / DBS-ON per session)

In [3]:
triplets = get_triplets()
trial_rows = [
    {
        "session": t.label or "",
        "participant": (t.label or "").split("_")[0],
        "DBS-OFF": sum(1 for r in load_results_for_triplet(t) if r.get("stim") == "off"),
        "DBS-ON": sum(1 for r in load_results_for_triplet(t) if r.get("stim") == "on"),
    }
    for t in triplets
]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=[r["session"] for r in trial_rows], y=[r["DBS-OFF"] for r in trial_rows],
    name="DBS-OFF", marker_color=COLOR_DBS_OFF, width=0.35,
    text=[r["DBS-OFF"] for r in trial_rows], textposition="outside",
    textfont=dict(size=FONT_SIZE_BASE, family=FONT_FAMILY),
))
fig.add_trace(go.Bar(
    x=[r["session"] for r in trial_rows], y=[r["DBS-ON"] for r in trial_rows],
    name="DBS-ON", marker_color=COLOR_DBS_ON, width=0.35,
    text=[r["DBS-ON"] for r in trial_rows], textposition="outside",
    textfont=dict(size=FONT_SIZE_BASE, family=FONT_FAMILY),
))
apply_thesis_style(
    fig, ThesisTheme.LIGHT, height=420,
    margin=dict(l=80, r=24, t=36, b=80), legend_y=-0.22,
)
fig.update_layout(
    barmode="group",
    bargap=0.28,
    xaxis=dict(title_text="", showgrid=False, tickfont=dict(size=FONT_SIZE_LABEL)),
    yaxis=dict(
        title=dict(text="test-set trial count",
                   font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY)),
        rangemode="tozero",
    ),
)
fig.write_image(str(OUT / 'fig_001_trial_count.png'), width=1000, height=420, scale=2)
fig.show()
print(
    "Fig 1 — Test-set trial counts per session, split by DBS condition. "
    f"Source: PSID triplet results (test split) for {', '.join(r['session'] for r in trial_rows)}. "
    f"Participants: {', '.join(sorted({r['participant'] for r in trial_rows}))}. "
    "Counts match the trial inventory table above after train/val/test splitting."
)

Fig 1 — Test-set trial counts per session, split by DBS condition. Source: PSID triplet results (test split) for PDI1_S2, PDI1_S4, PDI4_S2, PDI4_S3. Participants: PDI1, PDI4. Counts match the trial inventory table above after train/val/test splitting.


## Figs 2-5: PSD DBS comparison (4 sessions x 4 ECoG channels)

In [4]:
_PSD_SESSIONS = [
    ("PDI1_S2", "PDI1", 2),
    ("PDI1_S4", "PDI1", 4),
    ("PDI4_S2", "PDI4", 2),
    ("PDI4_S3", "PDI4", 3),
]
_ECOG_CHANNELS = ["ECOG_1", "ECOG_2", "ECOG_3", "ECOG_4"]

# Load block -> DBS mapping
_project_root = Path('.').resolve()
pq = _project_root / "data" / "participants.parquet"
block_dbs_df = pl.read_parquet(pq, hive_partitioning=True)
block_dbs = {
    (str(r["participant_id"]), int(r["session"]), int(r["block"])): int(r["dbs_stim"])
    for r in block_dbs_df.iter_rows(named=True)
}

data_root = _project_root / "data" / "resampled"
OFF_C, ON_C = COLOR_DBS_OFF, COLOR_DBS_ON

for label, pid, sess in _PSD_SESSIONS:
    pattern = f"sub-{pid}_ses-{sess}_task-copydraw_run-*_ieeg.parquet"
    run_files = sorted(data_root.glob(pattern))
    if not run_files:
        continue

    psds = {ch: {"off": [], "on": []} for ch in _ECOG_CHANNELS}
    freqs_ref = None

    for run_file in run_files:
        parts = run_file.stem.split("_")
        run_num = None
        for p in parts:
            if p.startswith("run-"):
                run_num = int(p[4:])
                break
        if run_num is None:
            continue
        dbs = block_dbs.get((pid, sess, run_num))
        if dbs is None:
            continue
        cond = "on" if dbs == 1 else "off"

        try:
            df = pl.read_parquet(run_file, columns=_ECOG_CHANNELS)
        except Exception:
            continue

        for ch in _ECOG_CHANNELS:
            if ch not in df.columns:
                continue
            signal = df[ch].to_numpy().astype(float)
            signal = signal[np.isfinite(signal)]
            if len(signal) < 2000:
                continue
            fs_native = 1000
            nperseg = min(len(signal), fs_native * 2)
            freqs, psd = welch(signal, fs=fs_native, nperseg=nperseg, noverlap=nperseg // 2)
            psd_db = 10 * np.log10(psd + 1e-20)
            if freqs_ref is None:
                freqs_ref = freqs
            psds[ch][cond].append(psd_db)

    if freqs_ref is None:
        continue

    freq_mask = freqs_ref <= 100
    freqs_ref_plot = freqs_ref[freq_mask]
    for ch in _ECOG_CHANNELS:
        for ck in ("off", "on"):
            psds[ch][ck] = [p[freq_mask] for p in psds[ch][ck]]

    ncols, nrows = 2, 2
    fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=False,
                        vertical_spacing=0.10, horizontal_spacing=0.10,
                        subplot_titles=list(_ECOG_CHANNELS))

    for ch_idx, ch in enumerate(_ECOG_CHANNELS):
        ri, ci = divmod(ch_idx, ncols)
        for cond_key, col_c, cond_label, line_dash in [
            ("off", OFF_C, "DBS-OFF", "dash"),
            ("on", ON_C, "DBS-ON", "solid"),
        ]:
            cond_psds = psds[ch][cond_key]
            if not cond_psds:
                continue
            mat = np.vstack(cond_psds)
            mean = mat.mean(axis=0)
            sem = mat.std(axis=0) / np.sqrt(len(mat))
            xb = np.concatenate([freqs_ref_plot, freqs_ref_plot[::-1]])
            yb = np.concatenate([mean + sem, (mean - sem)[::-1]])
            r_c, g_c, b_c = int(col_c[1:3], 16), int(col_c[3:5], 16), int(col_c[5:7], 16)
            fc = f"rgba({r_c},{g_c},{b_c},0.18)"
            fig.add_trace(go.Scatter(
                x=xb, y=yb, fill="toself", fillcolor=fc,
                line=dict(width=0), showlegend=False,
                name=cond_label, legendgroup=cond_label, hoverinfo="skip",
            ), row=ri+1, col=ci+1)
            fig.add_trace(go.Scatter(
                x=freqs_ref_plot, y=mean, mode="lines",
                line=dict(color=col_c, width=2.2, dash=line_dash),
                showlegend=(ch_idx == 0), name=cond_label, legendgroup=cond_label,
            ), row=ri+1, col=ci+1)

    psd_h = 720
    apply_thesis_style(fig, ThesisTheme.LIGHT, height=psd_h,
                       margin=dict(l=90, r=40, t=60, b=110), legend_y=-0.12)
    fig.update_annotations(font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY))
    for ch_idx in range(len(_ECOG_CHANNELS)):
        ri, ci = divmod(ch_idx, ncols)
        fig.update_xaxes(
            title_text="frequency (Hz)" if ri >= nrows - 1 else "",
            title_font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY),
            range=[0, 100], row=ri + 1, col=ci + 1,
        )
        fig.update_yaxes(
            title_text="PSD (dB/Hz)" if ci == 0 else "",
            title_font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY),
            row=ri + 1, col=ci + 1,
        )

    n_off_runs = sum(len(psds[ch]["off"]) for ch in _ECOG_CHANNELS) // len(_ECOG_CHANNELS)
    n_on_runs = sum(len(psds[ch]["on"]) for ch in _ECOG_CHANNELS) // len(_ECOG_CHANNELS)
    fig_idx = _PSD_SESSIONS.index((label, pid, sess)) + 2
    fig.write_image(
        str(OUT / f'fig_{fig_idx:03d}_psd_{label}.png'),
        width=1200, height=psd_h, scale=2,
    )
    fig.show()
    print(
        f"Fig {fig_idx} — Welch PSD (mean ± SEM) for {label} (participant {pid}, session {sess}), "
        f"computed on the raw 1000 Hz ECoG recordings under data/resampled/ "
        f"(sub-{pid}_ses-{sess}_task-copydraw_run-*_ieeg.parquet). "
        f"Pooled across {n_off_runs} DBS-OFF run(s) and {n_on_runs} DBS-ON run(s). "
        f"Each panel = one of {', '.join(_ECOG_CHANNELS)}; frequency range 0-100 Hz. "
        "Traces reveal the expected alpha/beta reduction under active DBS when present."
    )

Fig 2 — Welch PSD (mean ± SEM) for PDI1_S2 (participant PDI1, session 2), computed on the raw 1000 Hz ECoG recordings under data/resampled/ (sub-PDI1_ses-2_task-copydraw_run-*_ieeg.parquet). Pooled across 6 DBS-OFF run(s) and 6 DBS-ON run(s). Each panel = one of ECOG_1, ECOG_2, ECOG_3, ECOG_4; frequency range 0-100 Hz. Traces reveal the expected alpha/beta reduction under active DBS when present.


Fig 3 — Welch PSD (mean ± SEM) for PDI1_S4 (participant PDI1, session 4), computed on the raw 1000 Hz ECoG recordings under data/resampled/ (sub-PDI1_ses-4_task-copydraw_run-*_ieeg.parquet). Pooled across 5 DBS-OFF run(s) and 5 DBS-ON run(s). Each panel = one of ECOG_1, ECOG_2, ECOG_3, ECOG_4; frequency range 0-100 Hz. Traces reveal the expected alpha/beta reduction under active DBS when present.


Fig 4 — Welch PSD (mean ± SEM) for PDI4_S2 (participant PDI4, session 2), computed on the raw 1000 Hz ECoG recordings under data/resampled/ (sub-PDI4_ses-2_task-copydraw_run-*_ieeg.parquet). Pooled across 5 DBS-OFF run(s) and 5 DBS-ON run(s). Each panel = one of ECOG_1, ECOG_2, ECOG_3, ECOG_4; frequency range 0-100 Hz. Traces reveal the expected alpha/beta reduction under active DBS when present.


Fig 5 — Welch PSD (mean ± SEM) for PDI4_S3 (participant PDI4, session 3), computed on the raw 1000 Hz ECoG recordings under data/resampled/ (sub-PDI4_ses-3_task-copydraw_run-*_ieeg.parquet). Pooled across 5 DBS-OFF run(s) and 6 DBS-ON run(s). Each panel = one of ECOG_1, ECOG_2, ECOG_3, ECOG_4; frequency range 0-100 Hz. Traces reveal the expected alpha/beta reduction under active DBS when present.


## Fig 6: Tracing speed DBS comparison (velocity & acceleration, 4 sessions)

In [5]:
FS = 200

def _z_column_indices_for_outputs(triplet):
    res = load_split_results(RESULTS_ROOT, triplet.psid_variant, triplet.psid_run_ts, "test")
    names = channels_as_str_list(res.get("output_channels")) if res else []
    def _ix(exact):
        for i, n in enumerate(names):
            if str(n) == exact:
                return i
        return None
    iv = _ix("tracing_velocity_x")
    ia = _ix("tracing_acceleration_magnitude")
    return (iv if iv is not None else 0, ia if ia is not None else 1)

def _trial_z_column(z_arr, col_idx):
    if z_arr is None:
        return np.full(FS * 9, np.nan)
    za = np.asarray(z_arr, dtype=float)
    if za.ndim == 2:
        if col_idx >= za.shape[1]:
            return np.full(FS * 9, np.nan)
        za = za[:, col_idx]
    else:
        za = za.ravel()
    if len(za) < FS * 9:
        return np.pad(za, (0, FS * 9 - len(za)), constant_values=np.nan)
    return za[:FS * 9]

def _zscore_traces(z_off, z_on):
    def _norm(seq):
        if not seq:
            return seq
        pool = np.concatenate([np.asarray(a, dtype=float).ravel() for a in seq])
        pool = pool[np.isfinite(pool)]
        if not pool.size:
            return seq
        mu, sig = float(np.mean(pool)), float(np.std(pool))
        if sig < 1e-9:
            sig = 1.0
        return [(np.asarray(a, dtype=float) - mu) / sig for a in seq]
    return _norm(z_off), _norm(z_on)

triplets = get_triplets()
n_sessions_tr = len(triplets)
t = np.arange(FS * 9) / FS

# Layout: one row per session × 2 columns (velocity | acceleration). Drop per-cell
# subplot_titles to avoid repeated labels; use one session annotation per row instead.
fig = make_subplots(
    rows=n_sessions_tr, cols=2, shared_xaxes=True, shared_yaxes=False,
    vertical_spacing=0.07, horizontal_spacing=0.09,
)

trial_counts: list[tuple[str, int, int]] = []
for pi, tri in enumerate(triplets):
    trials = load_results_for_triplet(tri)
    if not trials:
        continue
    iv, ia = _z_column_indices_for_outputs(tri)

    def _collect(col_idx):
        off_l, on_l = [], []
        for row in trials:
            z = row.get("Z")
            if z is None:
                continue
            za = _trial_z_column(z, col_idx)
            stim = row.get("stim", "off")
            (off_l if stim == "off" else on_l).append(za)
        return off_l, on_l

    row = pi + 1
    show_legend_cell = (pi == 0)
    panel_counts = {"off": 0, "on": 0}
    for col_plot, col_idx in ((1, iv), (2, ia)):
        z_off, z_on = _collect(col_idx)
        z_off, z_on = _zscore_traces(z_off, z_on)
        for cond, col_c, clabel, key in (
            (z_off, OFF_C, "DBS-OFF", "off"),
            (z_on,  ON_C,  "DBS-ON",  "on"),
        ):
            if not cond:
                continue
            mat = np.vstack(cond)
            mean = np.nanmean(mat, axis=0)
            sem = np.nanstd(mat, axis=0) / np.sqrt(max(len(mat), 1))
            r_c, g_c, b_c = (
                int(col_c[1:3], 16), int(col_c[3:5], 16), int(col_c[5:7], 16),
            )
            ribbon = f"rgba({r_c},{g_c},{b_c},0.18)"
            fig.add_trace(go.Scatter(
                x=np.concatenate([t, t[::-1]]),
                y=np.concatenate([mean + sem, (mean - sem)[::-1]]),
                fill="toself", fillcolor=ribbon, line=dict(width=0),
                hoverinfo="skip", showlegend=False, legendgroup=clabel,
            ), row=row, col=col_plot)
            fig.add_trace(go.Scatter(
                x=t, y=mean, mode="lines",
                line=dict(color=col_c, width=2.2),
                showlegend=(show_legend_cell and col_plot == 1),
                name=clabel, legendgroup=clabel,
            ), row=row, col=col_plot)
            panel_counts[key] = len(cond)

    # In-panel session tag inside the velocity (left) cell.
    xref = "x" if row == 1 else f"x{(row - 1) * 2 + 1}"
    yref = "y" if row == 1 else f"y{(row - 1) * 2 + 1}"
    fig.add_annotation(
        x=0.02, y=0.95, xref=f"{xref} domain", yref=f"{yref} domain",
        text=f"<b>{tri.label}</b>", xanchor="left", yanchor="top",
        showarrow=False, font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY),
    )
    trial_counts.append((tri.label or "", panel_counts["off"], panel_counts["on"]))

# Column header annotations (once, at the top of the figure).
fig.add_annotation(
    x=0.5, y=1.02, xref="x domain", yref="y domain",
    text="<b>tracing_velocity_x</b>", showarrow=False, xanchor="center", yanchor="bottom",
    font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY),
)
fig.add_annotation(
    x=0.5, y=1.02, xref="x2 domain", yref="y2 domain",
    text="<b>tracing_acceleration_magnitude</b>", showarrow=False,
    xanchor="center", yanchor="bottom",
    font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY),
)

apply_thesis_style(
    fig, ThesisTheme.LIGHT,
    height=max(220 * n_sessions_tr + 140, 680),
    margin=dict(l=96, r=40, t=70, b=96),
    legend_y=-0.08,
)
fig.update_xaxes(range=[0, 9])
for rr in range(1, n_sessions_tr + 1):
    for cc in (1, 2):
        if rr == n_sessions_tr:
            fig.update_xaxes(title_text="time within trial (s)",
                             title_font=dict(size=FONT_SIZE_LABEL, family=FONT_FAMILY),
                             row=rr, col=cc)
        if cc == 1:
            fig.update_yaxes(title_text="z-score (trial-pooled)",
                             title_font=dict(size=FONT_SIZE_BASE, family=FONT_FAMILY),
                             row=rr, col=cc)

fig.write_image(str(OUT / 'fig_006_tracing_speed.png'),
                width=1100, height=max(220 * n_sessions_tr + 140, 680), scale=2)
fig.show()
print(
    "Fig 6 — Trial-averaged behavioural trajectories aligned to trial onset (0-9 s), "
    "pooled across all test trials and z-scored per condition. Left column: tracing_velocity_x. "
    "Right column: tracing_acceleration_magnitude. One row per session; trial counts "
    + ", ".join(f"{lab}: {n_off} OFF / {n_on} ON" for lab, n_off, n_on in trial_counts) + ". "
    "Data source: PSID test-split Z arrays."
)

Fig 6 — Trial-averaged behavioural trajectories aligned to trial onset (0-9 s), pooled across all test trials and z-scored per condition. Left column: tracing_velocity_x. Right column: tracing_acceleration_magnitude. One row per session; trial counts PDI1_S2: 20 OFF / 24 ON, PDI1_S4: 11 OFF / 21 ON, PDI4_S2: 12 OFF / 24 ON, PDI4_S3: 11 OFF / 25 ON. Data source: PSID test-split Z arrays.
